## **Bronze layer**


###### Pulls NBA data from the API and saves raw JSON to S3.

In [ ]:
!pip install nba_api boto3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.1 MB/s eta 0:00:00


In [ ]:
import os
os.environ["AWS_ACCESS_KEY_ID"] = "Your Access Key "
os.environ["AWS_SECRET_ACCESS_KEY"] = "Your Secret Access Key "
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"


In [ ]:
import json, time, boto3
from datetime import datetime
from nba_api.stats.endpoints import leaguegamefinder, teamestimatedmetrics

BUCKET = "nbapredictions-sthomas26-ncf"
SEASON = "2024-25"
RUN_TS = datetime.utcnow().strftime("%Y%m%dT%H%M%S")
s3 = boto3.client("s3")

def upload(data, key):
    s3.put_object(Bucket=BUCKET, Key=key, Body=json.dumps(data), ContentType="application/json")
    print(f"uploaded → s3://{BUCKET}/{key}")

print("Pulling game logs...")
games = leaguegamefinder.LeagueGameFinder(
    season_nullable=SEASON,
    league_id_nullable="00"
).get_dict()
upload(games, f"bronze/games/season={SEASON}/run={RUN_TS}/games.json")
time.sleep(1)

print("Pulling team metrics...")
stats = teamestimatedmetrics.TeamEstimatedMetrics(season=SEASON).get_dict()
upload(stats, f"bronze/team_stats/season={SEASON}/run={RUN_TS}/team_stats.json")

print("\nDone. Bronze layer populated.")

/tmp/ipykernel_7643/3849473126.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TS = datetime.utcnow().strftime("%Y%m%dT%H%M%S")


Pulling game logs...
uploaded → s3://nbapredictions-sthomas26-ncf/bronze/games/season=2024-25/run=20260514T004017/games.json
Pulling team metrics...
uploaded → s3://nbapredictions-sthomas26-ncf/bronze/team_stats/season=2024-25/run=20260514T004017/team_stats.json

Done. Bronze layer populated.
